<a href="https://colab.research.google.com/github/Anmolsharma44/tndrr-ai/blob/main/Task_1_Lora_FineTuning/A%5B1%5DLORA%2CQLORA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import random

services = ["haircut", "facial", "hair spa", "beard trim", "manicure"]
names = ["Rahul", "Amit", "Priya", "Neha", "Vikas", "Aniket", "Rohit", "Sneha"]
times = ["10:00 AM", "2:30 PM", "5:00 PM", "11:15 AM", "4:00 PM", "6:30 PM"]
days = ["today", "tomorrow", "Monday", "this Wednesday", "coming Friday"]

def generate_sample():
    service = random.choice(services)
    name = random.choice(names)
    time = random.choice(times)
    day = random.choice(days)

    templates = [
        f"Hi, please book a {service} for {name} at {time} {day}.",
        f"Hey, I want to schedule a {service} on {day} around {time}. My name is {name}.",
        f"Can you book {name} for a {service} for {day} at {time}?",
        f"Urgent: {service} booking needed for {name} for {time} {day}."
    ]
    prompt_text = random.choice(templates)
    json_output = {"service": service, "customer_name": name, "time": time, "date": day}
    return {"input": prompt_text, "target": json.dumps(json_output)}

# Dataset generate karo
train_data = [generate_sample() for _ in range(500)]
test_data = [generate_sample() for _ in range(60)]

print(f"Dataset Generated Successfully!")
print(f"Train samples: {len(train_data)} | Test samples: {len(test_data)}")
print("Sample Input:", train_data[0]["input"])
print("Sample Target:", train_data[0]["target"])

Dataset Generated Successfully!
Train samples: 500 | Test samples: 60
Sample Input: Hi, please book a hair spa for Priya at 4:00 PM coming Friday.
Sample Target: {"service": "hair spa", "customer_name": "Priya", "time": "4:00 PM", "date": "coming Friday"}


In [6]:
!pip install -q --upgrade torchao

import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Model ko float16 mein load karo
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

# 2. LoRA Config lagao
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
print("Model loaded and LoRA applied successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

W1117 17:51:49.294000 26769 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Model loaded and LoRA applied successfully!


Free Google Colab (T4 GPU) par isko run karne ke liye hum transformers, peft, bitsandbytes, aur trl
=================================================================================
Step 2: Libraries Install aur Model Load karna (QLoRA Setup)

In [7]:
!pip install -q torch transformers datasets peft bitsandbytes trl

from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded successfully!")

from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Qwen ke official chat template se format karo
def format_example(example):
    messages = [
        {"role": "system", "content": "Extract booking details into strict JSON format."},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["target"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_dataset = Dataset.from_list(train_data).map(format_example)

sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,          # Learning rate safe kar diya taaki weights corrupt na hon
    max_grad_norm=0.3,           # Gradient clipping (weights ko stable rakhne ke liye)
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    logging_steps=10,
    save_strategy="no",
    dataset_text_field="text",
    max_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("--- Training Started ---")
trainer.train()
print("--- Training Completed Successfully! ---")

Tokenizer loaded successfully!


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

--- Training Started ---


Step,Training Loss
10,1.710015
20,0.366495
30,0.170846
40,0.136621
50,0.124951
60,0.121669
70,0.119272
80,0.115123
90,0.115590


--- Training Completed Successfully! ---


Adversarial Testing ya Edge-Case Evaluation

In [10]:
# Kuch tricky aur edge-case samples khud se design karke add karte hain
adversarial_samples = [
    {
        "input": "Book a haircut for Rahul.",  # Missing time & date
        "target": '{"service": "haircut", "customer_name": "Rahul", "time": "N/A", "date": "N/A"}'
    },
    {
        "input": "Hi yaar kal subah 10 baje hair spa karwa dena Amit ke liye.",  # Hinglish mix
        "target": '{"service": "hair spa", "customer_name": "Amit", "time": "10:00 AM", "date": "tomorrow"}'
    },
    {
        "input": "Urgent booking for beard trim for Priya at 4 PM today, wait no make it 5 PM.",  # Conflicting time
        "target": '{"service": "beard trim", "customer_name": "Priya", "time": "5:00 PM", "date": "today"}'
    },
    {
        "input": "Schedule facial around noon next Monday.",  # Vague time description ("noon")
        "target": '{"service": "facial", "customer_name": "unknown", "time": "12:00 PM", "date": "next Monday"}'
    },
    {
        "input": "Book salon for Rohit at 3 PM.",  # Simple service name vs specific service
        "target": '{"service": "salon", "customer_name": "Rohit", "time": "3:00 PM", "date": "today"}'
    }
]

# Inko apne existing test_data ke end mein jod do
test_data.extend(adversarial_samples)
print(f"Total test samples now: {len(test_data)} (including 5 adversarial edge cases)")

Total test samples now: 65 (including 5 adversarial edge cases)


Step 3: LoRA Configuration (Hyperparameters & Reasoning)

yeh upar dataset bnaya


In [12]:
import json
import torch

model.eval()

def evaluate_with_comparison(eval_samples):
    # 1. Evaluate Base Model (Without LoRA)
    base_matches = 0
    print("Evaluating Base Model (Without LoRA)...")
    for sample in eval_samples:
        messages = [
            {"role": "system", "content": "Extract booking details into strict JSON format."},
            {"role": "user", "content": sample["input"]}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            with model.disable_adapter():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=60,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )

        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        pred_str = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        try:
            if json.loads(pred_str) == json.loads(sample["target"]):
                base_matches += 1
        except:
            pass

    base_acc = (base_matches / len(eval_samples)) * 100

    # 2. Evaluate Fine-Tuned Model (With LoRA) & Capture Failures
    ft_matches = 0
    failures = []
    print("Evaluating Fine-Tuned Model (With LoRA)...")
    for sample in eval_samples:
        messages = [
            {"role": "system", "content": "Extract booking details into strict JSON format."},
            {"role": "user", "content": sample["input"]}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=60,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        pred_json_str = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        try:
            pred_dict = json.loads(pred_json_str)
            target_dict = json.loads(sample["target"])

            if pred_dict == target_dict:
                ft_matches += 1
            else:
                if len(failures) < 3:
                    failures.append({"input": sample["input"], "target": sample["target"], "predicted": pred_json_str, "reason": "Field value / Adversarial mismatch"})
        except Exception:
            if len(failures) < 3:
                failures.append({"input": sample["input"], "target": sample["target"], "predicted": pred_json_str, "reason": "Invalid JSON syntax / Conversational filler"})

    ft_acc = (ft_matches / len(eval_samples)) * 100
    return base_acc, ft_acc, failures

# Run the comparison
base_accuracy, ft_accuracy, failure_cases = evaluate_with_comparison(test_data)

print(f"\n==========================================")
print(f"📊 COMPARISON RESULTS FOR REPORT:")
print(f"==========================================")
print(f"Base Model Exact JSON Accuracy     : {base_accuracy:.2f}%")
print(f"Fine-Tuned Model Exact JSON Accuracy: {ft_accuracy:.2f}%")
print(f"==========================================\n")

print("--- 3 Failure Cases for Report ---")
for idx, f in enumerate(failure_cases, 1):
    print(f"Failure #{idx}:\n  Input: {f['input']}\n  Expected: {f['target']}\n  Predicted: {f['predicted']}\n  Reason: {f['reason']}\n")

Evaluating Base Model (Without LoRA)...
Evaluating Fine-Tuned Model (With LoRA)...

📊 COMPARISON RESULTS FOR REPORT:
Base Model Exact JSON Accuracy     : 0.00%
Fine-Tuned Model Exact JSON Accuracy: 92.31%

--- 3 Failure Cases for Report ---
Failure #1:
  Input: Book a haircut for Rahul.
  Expected: {"service": "haircut", "customer_name": "Rahul", "time": "N/A", "date": "N/A"}
  Predicted: {"service": "haircut", "customer_name": "Rahul"}
  Reason: Field value / Adversarial mismatch

Failure #2:
  Input: Hi yaar kal subah 10 baje hair spa karwa dena Amit ke liye.
  Expected: {"service": "hair spa", "customer_name": "Amit", "time": "10:00 AM", "date": "tomorrow"}
  Predicted: {"service": "hair spa", "customer_name": "Amit", "date": "10", "time": "baje"}
  Reason: Field value / Adversarial mismatch

Failure #3:
  Input: Urgent booking for beard trim for Priya at 4 PM today, wait no make it 5 PM.
  Expected: {"service": "beard trim", "customer_name": "Priya", "time": "5:00 PM", "date": "tod

===============================================================================================================================================================


Step 4: Dataset Formatting aur Training (SFTTrainer)
Apne generated dataset ko Qwen ke chat template mein badal kar train karenge